# RMT-PPAD migration NB85 - Phase P6 (dataloader: lane_targets + lane_seg_mask)

**Purpose.** Verify the patched `YOLODataset` produces batches with the
right keys/shapes for the lane-only training path:
  - `img`           shape `(B, 3, 640, 640)`
  - `lane_targets`  shape `(B, 8, 78)`
  - `lane_seg_mask` shape `(B, 1, 640, 640)` (rasterized from `lane_targets`)
  - NO drivable channel

**Acceptance (appendix-path3 sec 9.6 + our P1 max_lanes=8 bump):**
shapes match the contract that P5's `MTDETRDLoss` expects.

**Wall time:** ~2-3 min (mmcv install + dataset build on synthetic data
- no Drive I/O for the small acceptance test).

**No real BDD images needed for this test** - `verify_dataloader.py`
synthesizes 10 random JPGs + matching `.pt` files in a temp dir so the
test stays hermetic. Real-data end-to-end check happens in P8.

### Cell 1: Mount Drive, install mmcv

In [1]:
import os, sys, subprocess
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import mmcv  # noqa: F401
    print(f'[ok] mmcv already installed: {mmcv.__version__}')
except ImportError:
    print('[install] mmcv ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
    import mmcv
    print(f'[ok] mmcv installed: {mmcv.__version__}')

import torch
print('torch', torch.__version__)
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
[install] mmcv ...
[ok] mmcv installed: 2.2.0
torch 2.10.0+cpu
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


### Cell 2: Smoke test the lane-mask rasterizer
Confirms `rasterize_lane_target_to_mask` produces correct shape and
non-empty output for a synthetic two-lane target. P6 + P7 share this code.

In [2]:
import sys, os
SCRIPT = 'stage2/rmt_ppad_migration/P6_dataset/tools/lane_mask_from_target.py'
log = os.path.join(LOG_DIR, 'NB85_lane_mask_smoke.log')
rc = run_streaming([sys.executable, '-u', SCRIPT], log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'lane_mask_from_target smoke rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P6_dataset/tools/lane_mask_from_target.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB85_lane_mask_smoke.log
[smoke] lane_mask_from_target starting
[smoke] mask shape (640, 640), dtype uint8, pixels-set 2297  (expect > 0)
[smoke] no-lane target -> empty mask OK
[smoke] PASS
[run_streaming] return_code=0


### Cell 3: Verify dataloader on a synthetic dataset
Asserts collated batch has the right `img`, `lane_targets`, and
`lane_seg_mask` shapes; lane_seg_mask has non-zero lane pixels.

In [3]:
import sys, os
VERIFY = 'stage2/rmt_ppad_migration/P6_dataset/tools/verify_dataloader.py'
log = os.path.join(LOG_DIR, 'NB85_verify_dataloader.log')
rc = run_streaming([sys.executable, '-u', VERIFY], log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'verify_dataloader rc={rc}; see {log}')

print('\n[P6 result] PASS')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P6_dataset/tools/verify_dataloader.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB85_verify_dataloader.log
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[run_streaming] still running; no child output yet. This usually means the first dataloader/model step is still working.
[smoke] building tiny dataset at /tmp/p6_ds_t210loqo
[smoke] instantiating MTDETRDataset (task=multi)

[P6 smoke]Scanning /tmp/p6_ds_t210loqo/labels/train...:   0%|          | 0/10 [00:00<?, ?it/s]
[P6 smoke]Scanning /tmp/p6_ds_t210loqo/labels/train... 10 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10/10 [00:00<00:00, 191.62it/s]
[P6 smoke